# Rap1 Transcription Factor Binding Site Classifier

This notebook implements a neural network classifier to predict whether a DNA sequence is a binding site for the yeast transcription factor Rap1.

In [1]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import sys
sys.path.insert(0, '..')
from nn.nn import NeuralNetwork
from nn.io import read_text_file, read_fasta_file
from nn.preprocess import sample_seqs, one_hot_encode_seqs

## Step 1: Read in the data

In [3]:
# Read positive examples (Rap1 binding sites)
pos_seqs = read_text_file('./data/rap1-lieb-positives.txt')
print(f"Number of positive sequences: {len(pos_seqs)}")
print(f"Length of positive sequences: {len(pos_seqs[0])}")
print(f"First 5 positive sequences:")
for i, seq in enumerate(pos_seqs[:5]):
    print(f"  {i+1}: {seq}")

Number of positive sequences: 137
Length of positive sequences: 17
First 5 positive sequences:
  1: ACATCCGTGCACCTCCG
  2: ACACCCAGACATCGGGC
  3: CCACCCGTACCCATGAC
  4: GCACCCATACATTACAT
  5: ACATCCATACACCCTCT


In [4]:
# Read negative examples (upstream sequences with no known binding)
neg_seqs = read_fasta_file('./data/yeast-upstream-1k-negative.fa')
print(f"Number of negative sequences: {len(neg_seqs)}")
print(f"Length of negative sequences (examples):")
for i in range(min(5, len(neg_seqs))):
    print(f"  Sequence {i+1}: {len(neg_seqs[i])} bp")

Number of negative sequences: 3163
Length of negative sequences (examples):
  Sequence 1: 1000 bp
  Sequence 2: 1000 bp
  Sequence 3: 1000 bp
  Sequence 4: 1000 bp
  Sequence 5: 1000 bp


## Step 2: Process negative sequences to match positive sequence length

In [ ]:
# Get the length of positive sequences (motif length)
motif_length = len(pos_seqs[0])
print(f"Motif length: {motif_length} bp")

# Extract subsequences from negative examples
# We'll sample random windows from each negative sequence
np.random.seed(42)
neg_seqs_sampled = []

for seq in neg_seqs:
    if len(seq) >= motif_length:
        # Random starting position
        start = np.random.randint(0, len(seq) - motif_length + 1)
        subseq = seq[start:start + motif_length]
        neg_seqs_sampled.append(subseq)

print(f"Number of negative subsequences: {len(neg_seqs_sampled)}")
print(f"First 5 negative subsequences:")
for i, seq in enumerate(neg_seqs_sampled[:5]):
    print(f"  {i+1}: {seq}")

## Step 3: Create labels and balance the dataset

In [ ]:
# Create labels
pos_labels = [True] * len(pos_seqs)
neg_labels = [False] * len(neg_seqs_sampled)

# Combine sequences and labels
all_seqs = pos_seqs + neg_seqs_sampled
all_labels = pos_labels + neg_labels

print(f"Initial dataset composition:")
print(f"  Positive samples: {sum(all_labels)}")
print(f"  Negative samples: {len(all_labels) - sum(all_labels)}")
print(f"  Ratio: {sum(all_labels) / len(all_labels) * 100:.2f}% positive")

In [ ]:
# Balance the dataset using sampling without replacement
balanced_seqs, balanced_labels = sample_seqs(all_seqs, all_labels)

print(f"\nBalanced dataset composition:")
print(f"  Positive samples: {sum(balanced_labels)}")
print(f"  Negative samples: {len(balanced_labels) - sum(balanced_labels)}")
print(f"  Ratio: {sum(balanced_labels) / len(balanced_labels) * 100:.2f}% positive")

### Sampling Scheme Explanation

The dataset is highly imbalanced with ~137 positive examples and ~4000+ negative examples. I used a **downsampling strategy**:

1. **Identify the minority class**: Positive sequences (137 samples)
2. **Downsample the majority class**: Randomly sample 137 negative sequences without replacement
3. **Result**: Balanced dataset with 137 positive and 137 negative samples (50-50 split)

**Why this approach?**
- **Avoids oversampling artifacts**: Oversampling would create duplicate samples and inflate performance metrics
- **Reduces overfitting**: Using fewer samples with balanced classes reduces the risk of the model learning spurious patterns
- **Faster training**: Smaller dataset trains faster
- **Fair classification**: Equal class weights ensure the model doesn't bias toward the majority class

## Step 4: One-hot encode sequences

In [ ]:
# One-hot encode all sequences
X = one_hot_encode_seqs(balanced_seqs)
y = np.array(balanced_labels, dtype=float).reshape(-1, 1)

print(f"Encoded data shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Feature dimension: {motif_length} bp * 4 nucleotides = {X.shape[1]}")

## Step 5: Split into training and validation sets

In [ ]:
# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"\nTraining set class distribution:")
print(f"  Positive: {np.sum(y_train)} ({np.sum(y_train)/len(y_train)*100:.1f}%)")
print(f"  Negative: {len(y_train) - np.sum(y_train)} ({(len(y_train)-np.sum(y_train))/len(y_train)*100:.1f}%)")

## Step 6: Create and train the classifier

In [ ]:
# Define classifier architecture
# Input: 17 nucleotides * 4 = 68 features
# Hidden layers for feature extraction
nn_arch = [
    {'input_dim': X_train.shape[1], 'output_dim': 32, 'activation': 'relu'},
    {'input_dim': 32, 'output_dim': 16, 'activation': 'relu'},
    {'input_dim': 16, 'output_dim': 1, 'activation': 'sigmoid'}
]

# Create classifier
classifier = NeuralNetwork(
    nn_arch=nn_arch,
    lr=0.01,
    seed=42,
    batch_size=16,
    epochs=100,
    loss_function='binary_cross_entropy'
)

print("Classifier created successfully")
print(f"Architecture: {X_train.shape[1]} -> 32 -> 16 -> 1")

In [ ]:
# Train the classifier
train_losses, val_losses = classifier.fit(X_train, y_train.T, X_val, y_val.T)

print(f"Training completed")
print(f"Final training loss: {train_losses[-1]:.6f}")
print(f"Final validation loss: {val_losses[-1]:.6f}")

## Step 7: Plot training and validation loss

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (Binary Cross-Entropy)', fontsize=12)
plt.title('Classifier Training and Validation Loss', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8: Evaluate classifier performance

In [ ]:
# Get predictions on validation set
y_hat_val = classifier.predict(X_val)

# Convert to binary predictions (threshold at 0.5)
y_pred_binary = (y_hat_val >= 0.5).astype(float)

# Calculate accuracy
accuracy = np.mean(y_pred_binary == y_val)
print(f"Validation Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Calculate precision, recall, and F1
tp = np.sum((y_pred_binary == 1) & (y_val == 1))
fp = np.sum((y_pred_binary == 1) & (y_val == 0))
fn = np.sum((y_pred_binary == 0) & (y_val == 1))
tn = np.sum((y_pred_binary == 0) & (y_val == 0))

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\nValidation Metrics:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1 Score: {f1:.4f}")
print(f"\nConfusion Matrix:")
print(f"  True Positives: {int(tp)}")
print(f"  False Positives: {int(fp)}")
print(f"  True Negatives: {int(tn)}")
print(f"  False Negatives: {int(fn)}")

## Hyperparameter and Loss Function Explanation

### Loss Function: Binary Cross-Entropy

**Why Binary Cross-Entropy?**
- This is a **binary classification problem** (binding site or not)
- BCE is the standard loss for binary classification with sigmoid output
- It directly optimizes for classification accuracy
- Formula: L = -[y*log(ŷ) + (1-y)*log(1-ŷ)]
- Heavily penalizes confident but incorrect predictions

### Network Architecture: Input → 32 → 16 → 1
- **Input layer**: 68 features (17 bp × 4 nucleotides)
- **Hidden layer 1**: 32 neurons with ReLU
  - Provides feature extraction and non-linearity
  - 32 is a reasonable intermediate size for this dataset size
- **Hidden layer 2**: 16 neurons with ReLU
  - Further dimensionality reduction
  - Helps prevent overfitting
- **Output layer**: 1 neuron with Sigmoid
  - Sigmoid outputs probability in [0, 1]
  - Suitable for binary classification

### Key Hyperparameters:
- **Learning Rate (0.01)**: 
  - Moderate value for stable convergence
  - Too high would cause oscillation
  - Too low would converge very slowly
- **Batch Size (16)**:
  - Small enough for stable gradients
  - Large enough for efficient training
  - Works well with ~220 training samples
- **Epochs (100)**:
  - Sufficient for convergence on this dataset
  - Prevents overfitting while ensuring learning

### Why These Choices Work for DNA Binding Prediction:
1. **ReLU activations**: Good for learning non-linear sequence patterns
2. **Sigmoid output**: Natural probability output for binding/non-binding decision
3. **Moderate architecture**: Not too deep (avoids vanishing gradients), not too shallow (sufficient capacity)
4. **Balanced dataset**: Equal class weights prevent bias toward majority class